### RUN in TPU

In [1]:
# CELL 0: Set Run Parameters
import os

os.environ["CACHE_LOOKBACK"] = "10"
os.environ["CACHE_START_DATE"] = "2026-01-01"
# Set to "True" to force identical splits regardless of size, or "False" to split dynamically
os.environ["DEBUG_MODE"] = "True"

In [2]:
# CELL 1
import time

start_time = time.time()

In [3]:
# CELL 2: Setup
import sys, os
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2"
    )
else:
    rl_root = Path("C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2")

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

In [4]:
# CELL 3: Imports
import torch
import numpy as np
import pandas as pd
from core.settings import TradingConfig, CacheConfig
from core.paths import OUTPUT_DIR
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar, get_chronological_splits
from data_pipeline.screener import UniverseScreener
from rl_discovery.oracle import RLOracle
from rl_discovery.environment import DiscoveryEnv
from rl_discovery.adapter import RLVRGymEnv
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.trainer import RolloutBuffer, PPOTrainer
from rl_discovery.validator import AgentEvaluator

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output



In [5]:
# CELL 4: Load Data & Environment Splits
print("Loading preprocessed data & AlphaCache...")
df_ohlcv, macro_df, features_df = load_processed_data()
config = TradingConfig()

df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

screener = UniverseScreener(
    df_close=df_close,
    features_df=features_df,
    macro_df=macro_df,
    trading_calendar=trading_calendar,
    config=config,
)

cache_file_path = OUTPUT_DIR / CacheConfig.get_filename()
print(f"cache_file_path:\n{cache_file_path}\n")
feature_cube = pd.read_parquet(cache_file_path)

oracle = RLOracle(screener, config)
oracle.precompute_reward_matrix(holding_period=config.holding_period)

# Generate Splits via the updated utility
cal_train, cal_val, cal_test = get_chronological_splits(
    trading_calendar=trading_calendar,
    feature_cube_dates=feature_cube.index.get_level_values("Date").unique(),
    holding_period=config.holding_period,
    min_dates_threshold=100,  # Adjust this threshold based on your minimum desired sequence length
)

env_train = DiscoveryEnv(
    feature_cube, oracle.reward_matrix, cal_train, config.holding_period
)
gym_train = RLVRGymEnv(env_train, macro_df)

env_val = DiscoveryEnv(
    feature_cube, oracle.reward_matrix, cal_val, config.holding_period
)
gym_val = RLVRGymEnv(env_val, macro_df)

env_test = DiscoveryEnv(
    feature_cube, oracle.reward_matrix, cal_test, config.holding_period
)
gym_test = RLVRGymEnv(env_test, macro_df)

Loading preprocessed data & AlphaCache...
cache_file_path:
C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\alpha_cache_10d_2026.parquet

[INFO] Entering Debug/Scarcity mode (DEBUG_MODE=True, Total Dates=118) less than min_dates_threshold of 100. Identical calendars will be used across all splits.


In [7]:
# Store the dataframes in a dictionary for easy iteration
dataframes_to_check = {
    "df_ohlcv": df_ohlcv,
    "macro_df": macro_df,
    "features_df": features_df,
    "df_close": df_close,
    "feature_cube": feature_cube,
}

for name, df in dataframes_to_check.items():
    print(f"=== Analysis for DataFrame: {name} ===")

    # Calculate total NaN values in the entire DataFrame
    total_nans = df.isna().sum().sum()
    print(f"Total NaN values: {total_nans}")

    # If there are NaN values, print the count per column
    if total_nans > 0:
        print("NaN count per column (only showing columns with NaNs):")
        nan_by_column = df.isna().sum()
        # Filter to show only columns that contain at least one NaN
        columns_with_nans = nan_by_column[nan_by_column > 0]
        print(columns_with_nans)
    else:
        print("No NaN values found.")

    print("-" * 50)

=== Analysis for DataFrame: df_ohlcv ===
Total NaN values: 19839724
NaN count per column (only showing columns with NaNs):
Adj Open     4959931
Adj High     4959931
Adj Low      4959931
Adj Close    4959931
dtype: int64
--------------------------------------------------
=== Analysis for DataFrame: macro_df ===
Total NaN values: 0
No NaN values found.
--------------------------------------------------
=== Analysis for DataFrame: features_df ===
Total NaN values: 35502151
NaN count per column (only showing columns with NaNs):
ATR                    4960734
Mom_21                 4993292
Consistency               6312
IR_63                  5064288
Ret_1d                 4961602
Range_Pos_20           4991628
Slope_P_5              4966614
Slope_V_5                 6312
RollingStalePct         197250
RollMedDollarVol       5156869
RollingSameVolCount     197250
dtype: int64
--------------------------------------------------
=== Analysis for DataFrame: df_close ===
Total NaN values: 495993

In [ ]:
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- THE V6 HYPERPARAMETERS ---

###########################
# SEEDS = [42, 12345, 987654, 555666777, 20240620, 314159265]
SEEDS = [42]
###########################

# 1 epoch = 1024 steps (shuffled out of chronological order) = 8 mini_batch
# 1 gradient update after 1 mini_batch
NUM_EPOCHS = 50
NUM_STEPS = 1024
MINI_BATCH_SIZE = 128  # Larger batches for smooth gradients
ENT_COEF = 0.001  # Allows the agent to reduce variance/uncertainty
EVAL_FREQ = 2  # Check Validation every 2 epochs
PATIENCE = 5  # Give it room to breathe

metadata_payload = {
    "SEEDS": SEEDS,
    "NUM_EPOCHS": NUM_EPOCHS,
    "NUM_STEPS": NUM_STEPS,
    "MINI_BATCH_SIZE": MINI_BATCH_SIZE,
    "ENT_COEF": ENT_COEF,
    "EVAL_FREQ": EVAL_FREQ,
    "PATIENCE": PATIENCE,
    "RANK_MAX_OFFSET": config.rank_max_width,
    "HOLDING_PERIOD": config.holding_period,
}

checkpoint_dir = OUTPUT_DIR / "model_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

global_best_sharpe = -np.inf
global_best_model_path = None
global_best_seed = None

print("\n--- [TASK 3] Executing PPO V6 Meta-Loop ---")

for seed in SEEDS:
    print(f"\n{'='*50}\n🚀 STARTING TRAINING SEED: {seed}\n{'='*50}")
    set_seed(seed)

    agent = AbsoluteZeroAgent(obs_dim=33, action_dim=13).to(device)
    trainer = PPOTrainer(agent, lr=3e-4, clip_coef=0.2, ent_coef=ENT_COEF)
    buffer = RolloutBuffer(
        num_steps=NUM_STEPS, obs_dim=33, action_dim=13, device=device
    )

    best_val_sharpe = -np.inf
    patience_counter = 0

    for epoch in range(NUM_EPOCHS):
        obs, _ = gym_train.reset()
        epoch_rewards = []

        for step in range(NUM_STEPS):
            obs_tensor = torch.tensor(obs, dtype=torch.float32).to(device)
            with torch.no_grad():
                action, logprob, _, value = agent.get_action_and_value(
                    obs_tensor.unsqueeze(0)
                )

            raw_action = action.cpu().numpy()[0]
            next_obs, reward, done, _, info = gym_train.step(raw_action)

            buffer.add(obs, action[0], logprob[0], reward, value[0], done)
            epoch_rewards.append(reward)

            obs = next_obs
            if done:
                obs, _ = gym_train.reset()

        obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            next_value = agent.get_value(obs_tensor).flatten()

        buffer.compute_advantages(next_value, next_done=done)
        diagnostics = trainer.update(
            buffer, update_epochs=4, mini_batch_size=MINI_BATCH_SIZE
        )
        buffer.step = 0

        if (epoch + 1) % EVAL_FREQ == 0:
            print(
                f"Epoch {epoch+1:03d} | Rew: {np.mean(epoch_rewards):.4f} | Loss: {diagnostics['total_loss']:.4f} | Ent: {diagnostics['entropy']:.4f}"
            )

            val_results = AgentEvaluator.evaluate(agent, gym_val, device)
            val_sharpe = val_results["sharpe_ratio"]
            val_ret = val_results["total_return"] * 100

            print(f"   -> [VAL] Sharpe: {val_sharpe:.3f} | Return: {val_ret:.2f}%")

            if val_sharpe > best_val_sharpe:
                best_val_sharpe = val_sharpe
                patience_counter = 0

                # Dynamic Filename
                dynamic_name = (
                    f"champion_seed_{seed}_ep_{epoch+1}_sh_{val_sharpe:.2f}.pt"
                )
                seed_best_path = checkpoint_dir / dynamic_name
                torch.save(agent.state_dict(), str(seed_best_path))
                print(f"   -> 🟢 NEW BEST! Saved as: {dynamic_name}")

                # Track Global
                if val_sharpe > global_best_sharpe:
                    global_best_sharpe = val_sharpe
                    global_best_model_path = seed_best_path
                    global_best_seed = seed
            else:
                patience_counter += 1
                print(
                    f"   -> 🔴 No improvement. Patience: {patience_counter}/{PATIENCE}"
                )

            if patience_counter >= PATIENCE:
                print(f"🛑 EARLY STOPPING triggered for Seed {seed}")
                break

print(f"\n{'*'*50}")
print(f"🏆 GLOBAL CHAMPION CROWNED 🏆")
print(f"Best Seed:   {global_best_seed}")
print(f"Best Sharpe: {global_best_sharpe:.3f}")
print(f"File Path:   {global_best_model_path.name}")
print(f"{'*'*50}\n")

In [ ]:
# CELL 6: OOS Deterministic Evaluation & Blotter Review
import json

print("\n--- [TASK 4] Walk-Forward Deterministic Evaluation (OOS) ---")

# 1. Load the Ultimate Champion Model
champion_agent = AbsoluteZeroAgent(obs_dim=33, action_dim=13).to(device)
champion_agent.load_state_dict(torch.load(global_best_model_path))
print(f"[LOADED] Champion weights restored from: {global_best_model_path.name}")

# 2. Build Metadata Payload
metadata_payload["BEST_SEED"] = global_best_seed
metadata_payload["CHAMPION_FILE"] = global_best_model_path.name

# 3. Evaluate using the V6 Glass Box
results = AgentEvaluator.evaluate(
    champion_agent, gym_test, device, detailed_log=True, metadata=metadata_payload
)

print(f"\n[OOS TEST METRICS]")
print(f"Total Cumulative Return: {results['total_return']*100:.2f}%")
print(f"Sharpe Ratio (Annualized): {results['sharpe_ratio']:.3f}")
print(f"Total Trading Steps: {results['steps']}")

# 4. Save results
import pickle

results_save_path = output_path / "oos_evaluation_results_v6.pkl"
with open(results_save_path, "wb") as f:
    pickle.dump(results, f)
print(f"\n[SAVED] Complete V6 evaluation dictionary saved to {results_save_path}")

# 5. Print the Glass Box Blotter (First 2 days)
print("\n--- V6 GLASS BOX BLOTTER INSPECTION (First 2 Steps) ---")
blotter = results["blotter"]
for i in range(2):
    print(json.dumps(blotter[i], indent=2))

In [ ]:
# CELL 7: Timing
end_time = time.time()
print(f"Time: {time.strftime('%H:%M:%S', time.gmtime(end_time - start_time))}")